<a href="https://colab.research.google.com/github/ruslanmv/AutoSelf-Consistent-Multi-Agent-Platform/blob/master/AutoSelf_Consistent_Multi_Agent_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoSelf: Reproducible Experiments Notebook
This single notebook reproduces all figures/tables for the paper. It:
1) clones the repo, installs deps, and loads configs/seeds;
2) runs both experiments across seeds and ablations;
3) writes standardized CSVs to `results/`; regenerates figures;
4) computes medians and 95% CIs;
5) exports LaTeX macros to `results/latex_values.tex`; and
6) verifies every manuscript-cited artifact path exists (Artifact Index).

In [ ]:
import os
import shutil
import sys
from pathlib import Path
import subprocess

# --- 1. Define repository details ---
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/ruslanmv/AutoSelf-Consistent-Multi-Agent-Platform.git')
# --- Add the specific branch/tag you want to clone ---
BRANCH_NAME = 'v2.0.0'
REPO_DIR = Path('repo').resolve()

# --- 2. Clone the repository or switch to the correct branch ---
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print(f'Repository already exists at {REPO_DIR}.')
    # If the repo is already there, just switch to the correct branch/tag
    os.chdir(REPO_DIR)
    print(f'Switching to branch/tag: {BRANCH_NAME}')
    # Using !git for notebook compatibility
    !git checkout {BRANCH_NAME}
    !git pull
else:
    print(f'Cloning repository from {REPO_URL} on branch {BRANCH_NAME}...')
    # --- Add the -b flag to clone the specific branch/tag ---
    result = subprocess.run(
        ['git', 'clone', '-b', BRANCH_NAME, REPO_URL, str(REPO_DIR)], 
        capture_output=True, 
        text=True
    )
    if result.returncode != 0:
        print("Error cloning repository:", result.stderr, file=sys.stderr)
        raise ChildProcessError("Git clone failed.")
    print('Clone successful.')
    # Change the current working directory to the repository root
    os.chdir(REPO_DIR)

# --- 3. Verify the current version ---
print(f'\nCurrent working directory: {os.getcwd()}')
print('\nRepository head:')
!git log -1 --pretty=oneline

In [ ]:
import sys
import subprocess

# --- 2. Install Dependencies ---
# Use sys.executable to ensure pip is from the correct environment.
print('Installing dependencies...')
command = [
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt',
    'python-dotenv', 'pyyaml', 'pandas', 'numpy', 'matplotlib', 'seaborn',
    'beeai_framework==0.1.23'
]

# The installation of beeai_framework is optional; ignore failures.
try:
    # First, try installing everything together.
    subprocess.run(command, check=True)
    print('All dependencies installed successfully.')
except subprocess.CalledProcessError:
    print('Initial installation failed, retrying without optional packages...')
    # If it fails, try again without the optional package.
    command.pop() # Remove 'beeai_framework==0.1.23'
    try:
        subprocess.run(command, check=True)
        print('Core dependencies installed successfully. Optional beeai_framework failed.')
    except subprocess.CalledProcessError as e:
        print(f"Fatal error installing core dependencies: {e}", file=sys.stderr)
        raise


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# --- 3. Load Environment and Set Paths ---
print('Loading .env file if present...')
load_dotenv() # Loads variables from a .env file into the environment

# Check for optional LLM credentials (experiments can run offline without them)
print('Checking for optional LLM credentials...')
LLM_ENABLED = all(os.getenv(k) for k in ['WATSONX_API_KEY', 'PROJECT_ID', 'WATSONX_URL'])
print(f'LLM Status: {"ENABLED" if LLM_ENABLED else "DISABLED"}')

# Set and create artifact directories using environment variables for portability
RESULTS_DIR = Path(os.environ.setdefault('AUTOSELF_RESULTS_DIR', 'results'))
CONFIG_DIR = Path(os.environ.setdefault('AUTOSELF_CONFIG_DIR', 'configs'))

print(f'Results will be saved to: {RESULTS_DIR.resolve()}')
print(f'Configurations loaded from: {CONFIG_DIR.resolve()}')

# Create the results directory if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Directories configured.')

In [ ]:
import yaml
from pathlib import Path

# --- 4. Load Configuration Files ---
print('Loading YAML configuration files...')
try:
    cfg_autoself = yaml.safe_load((CONFIG_DIR / 'autoself.yml').read_text())
    cfg_world    = yaml.safe_load((CONFIG_DIR / 'world.yml').read_text())
    cfg_base     = yaml.safe_load((CONFIG_DIR / 'baselines.yml').read_text())
    seeds        = yaml.safe_load(Path('seeds.yaml').read_text())
    print('Successfully loaded: autoself.yml, world.yml, baselines.yml, seeds.yaml')

    # Print key values to verify configs
    p_grid = cfg_base.get('contention', {}).get('p_values', 'Not found')
    num_seeds = len(seeds.get('hazards_failures', {}).get('seeds', []))
    print(f'Contention p-grid: {p_grid}')
    print(f'Number of seeds for hazards/failures experiment: {num_seeds}')
except FileNotFoundError as e:
    print(f"Error: A configuration file is missing: {e}", file=sys.stderr)
    raise
except yaml.YAMLError as e:
    print(f"Error: Failed to parse a YAML file: {e}", file=sys.stderr)
    raise

In [ ]:
import json
import hashlib
import platform
import subprocess
from pathlib import Path

# --- 4b. Create Reproducibility Manifest ---
print('Creating reproducibility manifest...')

def get_git_hash():
    return subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()

def file_checksum(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    'git_commit_hash': get_git_hash(),
    'python_version': f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}',
    'platform': platform.platform(),
    'configs': {
        'autoself.yml': file_checksum(CONFIG_DIR / 'autoself.yml'),
        'world.yml': file_checksum(CONFIG_DIR / 'world.yml'),
        'baselines.yml': file_checksum(CONFIG_DIR / 'baselines.yml'),
        'seeds.yaml': file_checksum('seeds.yaml'),
    },
    'llm_enabled_in_env': LLM_ENABLED
}

manifest_path = RESULTS_DIR / 'repro_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f'Wrote manifest to {manifest_path}')

In [ ]:
import subprocess
import shlex
import sys

# --- 5. Run Experiments ---

def run_experiment(script_name, cli_args):
    """Helper function to run an experiment script with given arguments."""
    command = f'python {script_name} {cli_args}'
    print(f'---> Running command: {command}')
    
    result = subprocess.run(shlex.split(command), capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f'\n--- !!! WARNING: Experiment run returned non-zero exit code {result.returncode} !!! ---',
              file=sys.stderr)
        print("--- STDOUT ---", result.stdout, file=sys.stderr)
        print("--- STDERR ---", result.stderr, file=sys.stderr)
        print("---------------------------------------------------------------------", file=sys.stderr)
    return result.returncode

# Determine LLM flag based on environment
llm_flag = 'on' if LLM_ENABLED else 'off'
print(f'\nRunning experiments with --llm {llm_flag}')

# Run Experiment 1: Hazards and Failures Workflow
print('\n--- Starting Experiment 1: Hazards & Failures ---')
hazard_seeds = seeds.get('hazards_failures', {}).get('seeds', [])
for seed_val in hazard_seeds:
    args = f'--config-dir {CONFIG_DIR} --seeds seeds.yaml --seed-override {seed_val} --mode full --llm {llm_flag}'
    run_experiment('first_experiment.py', args)
print('--- Experiment 1 runs complete. ---\n')

# Run Experiment 2: Resource Contention Benchmark
print('--- Starting Experiment 2: Resource Contention (with ablations) ---')
ablation_modes = ['rules-only', 'sim-only', 'full']
for mode in ablation_modes:
    args = f'--config-dir {CONFIG_DIR} --seeds seeds.yaml --mode {mode} --llm {llm_flag}'
    run_experiment('second_experiment.py', args)
print('--- Experiment 2 runs complete. ---')


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# --- 6. Analyze Results ---
print('Aggregating results, computing medians and 95% CIs...')

def load_csv(filename):
    path = RESULTS_DIR / filename
    if not path.exists():
        print(f'Warning: Result file not found: {path}', file=sys.stderr)
        return pd.DataFrame()
    return pd.read_csv(path)

def calculate_ci95(data):
    """Calculates median and 95% confidence interval using percentiles."""
    clean_data = np.asarray(data.dropna())
    if clean_data.size < 2:
        return (np.nan, np.nan, np.nan)
    median = np.median(clean_data)
    ci_low, ci_high = np.percentile(clean_data, [2.5, 97.5])
    return (median, ci_low, ci_high)

# Load all result files
throughput_df = load_csv('throughput.csv')
makespan_df   = load_csv('makespan.csv')
conflicts_df  = load_csv('conflicts.csv')
ablations_df  = load_csv('ablations.csv')
overhead_df   = load_csv('overhead.csv') # New

# Aggregate throughput results (Experiment 2)
if not throughput_df.empty:
    agg_df = (throughput_df.groupby(['scenario', 'p'])
                .throughput_tpc.apply(lambda s: pd.Series(calculate_ci95(s), index=['median', 'ci_low', 'ci_high']))
                .reset_index())
    agg_path = RESULTS_DIR / 'throughput_summary.csv'
    agg_df.to_csv(agg_path, index=False)
    print(f'Wrote aggregated throughput summary to {agg_path}')
    display(agg_df.head())

    median_df = agg_df.pivot(index='p', columns='scenario', values='median')
    if {'autoself', 'baseline'}.issubset(set(median_df.columns)):
        median_df['delta_pct'] = 100.0 * (median_df['autoself'] - median_df['baseline']) / median_df['baseline']
        delta_path = RESULTS_DIR / 'throughput_delta.csv'
        median_df.to_csv(delta_path)
        print(f'Wrote throughput delta comparison to {delta_path}')
        display(median_df)
else:
    print('Throughput analysis skipped: throughput.csv is empty or missing.')

# Aggregate hazards/failures results (Experiment 1)
if not makespan_df.empty:
    hazards_summary_df = makespan_df[makespan_df['scenario'].isin(['nominal', 'hazard', 'failure'])]
    if not hazards_summary_df.empty:
        summary = hazards_summary_df.groupby('scenario').agg(
            makespan_median=('makespan_s', 'median'),
            makespan_ci_low=('makespan_s', lambda x: np.percentile(x, 2.5)),
            makespan_ci_high=('makespan_s', lambda x: np.percentile(x, 97.5)),
            unsafe_entries_total=('unsafe_entries', 'sum')
        ).reset_index()
        summary_path = RESULTS_DIR / 'hazards_summary.csv'
        summary.to_csv(summary_path, index=False)
        print(f'Wrote hazards summary to {summary_path}')
        display(summary)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import seaborn as sns

# --- 7. Generate Figures ---
print('Regenerating key figures from result CSVs...')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 8})

# Figure for Experiment 2: Throughput vs. Contention
throughput_path = RESULTS_DIR / 'throughput.csv'
if throughput_path.exists():
    throughput_df = pd.read_csv(throughput_path)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.lineplot(data=throughput_df, x='p', y='throughput_tpc', hue='scenario', marker='o', errorbar=('ci', 95), ax=ax)
    ax.set_title('Throughput vs. Contention Probability (Median with 95% CI)')
    ax.set_xlabel('Resource Contention Probability (p)')
    ax.set_ylabel('Throughput (tasks/cycle)')
    ax.grid(True, linestyle='--', alpha=0.6)
    
    for ext in ['pdf', 'png']:
        output_path = RESULTS_DIR / f'throughput_plot.{ext}'
        plt.savefig(output_path, bbox_inches='tight')
        print(f'Saved figure to {output_path}')
    plt.show()
else:
    print('Plotting skipped: throughput.csv not found.', file=sys.stderr)

# Figures for Experiment 1: Mission Timelines
for case in ['nominal', 'hazard', 'failure']:
    timeline_path = RESULTS_DIR / f'timeline_{case}.csv'
    if timeline_path.exists():
        timeline_df = pd.read_csv(timeline_path)
        fig, ax1 = plt.subplots(figsize=(8, 4))
        ax2 = ax1.twinx()
        
        sns.lineplot(data=timeline_df, x='time_s', y='tasks_completed', ax=ax1, color='b', label='Tasks Completed', drawstyle='steps-post')
        sns.lineplot(data=timeline_df, x='time_s', y='power_draw_w', ax=ax2, color='r', label='Power (W)', alpha=0.6)
        
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Cumulative Tasks Completed', color='b')
        ax2.set_ylabel('Site Power Draw (W)', color='r')
        ax1.tick_params(axis='y', labelcolor='b')
        ax2.tick_params(axis='y', labelcolor='r')
        ax1.grid(True, linestyle='--', alpha=0.6)
        plt.title(f'Mission Timeline: {case.capitalize()} Scenario')
        
        for ext in ['pdf', 'png']:
            output_path = RESULTS_DIR / f'{case}_timeline.{ext}'
            plt.savefig(output_path, bbox_inches='tight')
            print(f'Saved figure to {output_path}')
        plt.show()
    else:
        print(f'Plotting skipped: {timeline_path.name} not found.', file=sys.stderr)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- 8. Export LaTeX Macros ---
print('Exporting key results as LaTeX macros for manuscript...')

macros = []
def format_tex_macro(name, value):
    return f'\newcommand{{\{name}}}{{{value}}}'

# Load all summary files
summary_path = RESULTS_DIR / 'throughput_summary.csv'
delta_path = RESULTS_DIR / 'throughput_delta.csv'
hazards_path = RESULTS_DIR / 'hazards_summary.csv'
overhead_path = RESULTS_DIR / 'overhead.csv'

# Throughput macros (p=0.5)
if summary_path.exists():
    df = pd.read_csv(summary_path)
    row = df[(df['scenario'] == 'autoself') & (abs(df['p'] - 0.5) < 1e-6)]
    if not row.empty: 
        m, lo, hi = row.iloc[0][['median', 'ci_low', 'ci_high']]
        macros.extend([
            format_tex_macro('ThroughputMedianPfive', f"{m:.2f}"),
            format_tex_macro('ThroughputCIloPfive', f"{lo:.2f}"),
            format_tex_macro('ThroughputCIhiPfive', f"{hi:.2f}")
        ])

# Throughput delta macro (p=0.5)
if delta_path.exists():
    df = pd.read_csv(delta_path)
    row = df[abs(df['p'] - 0.5) < 1e-6]
    if not row.empty and 'delta_pct' in row.columns:
        macros.append(format_tex_macro('MakespanDeltaPfive', f"{row.iloc[0]['delta_pct']:.1f}\%" ))

# Hazards/Failures makespan and safety macros
if hazards_path.exists():
    df = pd.read_csv(hazards_path).set_index('scenario')
    if 'nominal' in df.index and 'hazard' in df.index and 'failure' in df.index:
        nominal_m = df.loc['nominal', 'makespan_median']
        macros.extend([
            format_tex_macro('MakespanDeltaHazard', f"{100*(df.loc['hazard','makespan_median']/nominal_m - 1):.1f}\%" ),
            format_tex_macro('MakespanDeltaFailure', f"{100*(df.loc['failure','makespan_median']/nominal_m - 1):.1f}\%" ),
            format_tex_macro('UnsafeEntriesZero', str(int(df['unsafe_entries_total'].sum())))
        ])

# Overhead macros
if overhead_path.exists():
    df = pd.read_csv(overhead_path)
    macros.extend([
        format_tex_macro('OverheadRulesMsMedian', f"{df['rules_ms'].median():.1f}"),
        format_tex_macro('OverheadSimMsMedian', f"{df['sim_ms'].median():.1f}"),
        format_tex_macro('OverheadLLMMsMedian', f"{df['llm_ms'].median():.1f}"),
        format_tex_macro('OverheadTotalMsMedian', f"{df['total_verif_ms'].median():.1f}")
    ])

if macros:
    output_file = RESULTS_DIR / 'latex_values.tex'
    output_content = '\n'.join(macros)
    output_file.write_text(output_content)
    print(f'Wrote {len(macros)} macros to {output_file}')
    print('--- File Content ---')
    print(output_content)
else:
    print('No data found to generate LaTeX macros.')

In [ ]:
from pathlib import Path

# --- 9. Final Artifact Verification ---
print('Verifying existence of all manuscript-cited artifacts...')

required_artifacts = [
    # Configs & Manifest
    CONFIG_DIR / 'autoself.yml',
    CONFIG_DIR / 'world.yml',
    CONFIG_DIR / 'baselines.yml',
    Path('seeds.yaml'),
    RESULTS_DIR / 'repro_manifest.json',

    # Raw Results
    RESULTS_DIR / 'throughput.csv',
    RESULTS_DIR / 'makespan.csv',
    RESULTS_DIR / 'conflicts.csv',
    RESULTS_DIR / 'ablations.csv',
    RESULTS_DIR / 'overhead.csv',
    RESULTS_DIR / 'timeline_nominal.csv',
    RESULTS_DIR / 'timeline_hazard.csv',
    RESULTS_DIR / 'timeline_failure.csv',

    # Summaries
    RESULTS_DIR / 'throughput_summary.csv',
    RESULTS_DIR / 'throughput_delta.csv',
    RESULTS_DIR / 'hazards_summary.csv',

    # Figures
    RESULTS_DIR / 'throughput_plot.pdf',
    RESULTS_DIR / 'throughput_plot.png',
    RESULTS_DIR / 'nominal_timeline.pdf',
    RESULTS_DIR / 'nominal_timeline.png',
    RESULTS_DIR / 'hazard_timeline.pdf',
    RESULTS_DIR / 'hazard_timeline.png',
    RESULTS_DIR / 'failure_timeline.pdf',
    RESULTS_DIR / 'failure_timeline.png',

    # LaTeX Exports
    RESULTS_DIR / 'latex_values.tex',
]

missing_artifacts = []
for path in required_artifacts:
    if not Path(path).exists():
        status = '❌ MISSING'
        missing_artifacts.append(str(path))
    else:
        status = '✅ OK'
    print(f'{status}: {path}')

if missing_artifacts:
    error_message = f'CRITICAL: {len(missing_artifacts)} required artifacts are missing: {missing_artifacts}'
    print(f'\n{error_message}', file=sys.stderr)
    raise FileNotFoundError(error_message)

print('\nArtifact Index Check Passed: All required artifacts are present.')